# 01 — Build the utility registry

Discover which platform a utility's outage map uses, test-fetch it, add it to
`registry.json`, then run a full collection pass to confirm. Run
`00_setup.ipynb` first if you have not. Run the cells below top to bottom.

## Setup

In [ ]:
import sys, json
from pathlib import Path

# Make the repo-root modules importable from this notebook (it lives in notebooks/).
root = Path.cwd()
while not (root / "config.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
print("repo root:", root)

import pandas as pd
import config
from discover import classify_outage_map
from kubra import KubraAdapter
from arcgis import ArcGISAdapter

ADAPTERS = {"kubra": KubraAdapter, "arcgis": ArcGISAdapter}

## Step 1 — Classify a utility's outage map

Paste the public outage-map URL below. This is a heuristic first pass — confirm
the exact endpoints with browser DevTools (Network tab), as the output explains.

In [ ]:
outage_map_url = "https://outagemap.example-utility.com"   # <-- change me

discovery = classify_outage_map(outage_map_url)
print(discovery.summary())

## Step 2 — Test-fetch a candidate registry entry

Fill in the `config` block from what DevTools showed you, then test the adapter
*before* committing the entry. Edit `platform` and the `config` keys to match
Step 1 (KUBRA keys are shown; for ArcGIS use `service_url` / `customers_field`).

In [ ]:
# Edit this candidate entry, then run to test it.
candidate = {
    "utility_id": "EXAMPLE_UTIL",
    "utility_name": "Example Utility",
    "platform": "kubra",                 # "kubra" or "arcgis"
    "states": ["IL"],
    "customers_served": 0,
    "enabled": True,
    "config": {
        # KUBRA -- replace with the URLs DevTools showed:
        "metadata_url": "https://CHANGE-ME.kubra.io/.../metadata.json",
        "tile_url_template": "https://CHANGE-ME.kubra.io/data/{data_dir}/public/cluster-1/{quadkey}.json",
        "min_zoom": 1,
        "max_zoom": 14,
        "seed_quadkeys": ["032", "033"],
        # ArcGIS alternative:
        # "service_url": "https://CHANGE-ME/arcgis/rest/services/Outages/MapServer/0",
        # "customers_field": "CustomersAffected",
    },
}

adapter = ADAPTERS[candidate["platform"]]()
records = adapter.fetch(candidate["utility_id"], candidate["config"])
print(f"{len(records)} records fetched")
pd.DataFrame(r.as_dict() for r in records).head(20)

## Step 3 — Add the entry to `registry.json`

Once the test fetch looks right, add (or replace) the entry in the registry.

In [ ]:
def upsert_registry_entry(entry, registry_path=config.REGISTRY_PATH):
    # Add a new utility, or replace an existing one with the same utility_id.
    entries = json.loads(Path(registry_path).read_text())
    entries = [e for e in entries if e.get("utility_id") != entry["utility_id"]]
    entries.append(entry)
    Path(registry_path).write_text(json.dumps(entries, indent=2))
    enabled = sum(bool(e.get("enabled")) for e in entries)
    print(f"registry.json now has {len(entries)} entries ({enabled} enabled)")

upsert_registry_entry(candidate)

## Step 4 — Run a full collection pass

Confirm the whole registry collects cleanly. This writes one snapshot to
`data/snapshots/`.

In [ ]:
from collector import collect_once

snapshot_path = collect_once()
snap = pd.read_parquet(snapshot_path)
print(f"\n{len(snap)} records in {snapshot_path}")
snap.groupby("utility_id")["customers_out"].agg(["count", "sum"])

---

Repeat Steps 1-3 for each utility, **biggest first** -- the largest ~100-150
utilities cover most US customers. When the registry is solid, schedule
`collector.py` (cron, or `.github/workflows/collect.yml`) so snapshots
accumulate, then move to `02_aggregate_and_verify.ipynb`.